# Defensive-line get-off submission

This notebook recomputes the 2023–24 study in `writeup/getoff.md`. It does not run the closed receiver study and it does not read 2025 get-off.

Three paths stay separate:

- `CODE_ROOT`: source and `config/getoff_v1.yaml`. Treat as read-only.
- `DATA_ROOT`: the competition CSV directory. Treat as read-only.
- `WORK_ROOT`: tables, JSON, and figures. This is the only path the study writes.

## Local

```bash
export BDB_CODE_ROOT=/path/to/this/repository
export BDB_DATA_ROOT=/path/to/nfl-big-data-bowl-2027
export BDB_WORK_ROOT=/path/to/an/empty/writable/directory
```

`BDB_DATA_ROOT` must contain `players.csv` directly, not a parent folder.

## Kaggle

Kaggle execution of this notebook has not been run from this machine. To run it there:

1. Add the competition data, which mounts at `/kaggle/input/nfl-big-data-bowl-2027`.
2. Upload this repository as a Kaggle dataset so `src/` and `config/` are available. Input datasets are read-only.
3. In the notebook settings, or in the first cell, set `BDB_CODE_ROOT` to that dataset directory.
4. Leave `BDB_WORK_ROOT` unset. The study then writes to `/kaggle/working/getoff-study`.
5. The Kaggle image needs `duckdb`, `pyyaml`, `pandas`, `numpy`, `scikit-learn`, and `matplotlib`. If one is missing, install it into the notebook environment before the run. Do not install the package by writing into `CODE_ROOT`.

The next cell inserts `CODE_ROOT/src` on `sys.path`. No editable install is required.


In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
if os.environ.get("BDB_CODE_ROOT"):
    CODE_ROOT = Path(os.environ["BDB_CODE_ROOT"]).resolve()
else:
    CODE_ROOT = next(path for path in candidates if (path / "config" / "getoff_v1.yaml").exists())

src = str(CODE_ROOT / "src")
sys.path.insert(0, src)
os.environ["PYTHONPATH"] = src + os.pathsep + os.environ.get("PYTHONPATH", "")

from bdb2027.getoff import resolve_paths, validate_inputs

DATA_ROOT = Path(os.environ["BDB_DATA_ROOT"]).resolve() if os.environ.get("BDB_DATA_ROOT") else None
WORK_ROOT = Path(os.environ["BDB_WORK_ROOT"]).resolve() if os.environ.get("BDB_WORK_ROOT") else None
CODE_ROOT, DATA_ROOT, WORK_ROOT = resolve_paths(CODE_ROOT, DATA_ROOT, WORK_ROOT)
validate_inputs(CODE_ROOT, DATA_ROOT)
print("CODE_ROOT", CODE_ROOT)
print("DATA_ROOT", DATA_ROOT)
print("WORK_ROOT", WORK_ROOT)


In [ ]:
from bdb2027.getoff import common_penalty_diagnostic, load_spec, run
import pandas as pd

summary = run(CODE_ROOT, DATA_ROOT, WORK_ROOT)
ledger = pd.read_csv(WORK_ROOT / "player_ledger.csv")
diagnostic = common_penalty_diagnostic(ledger, load_spec(CODE_ROOT))
figures = [WORK_ROOT / "presentation" / name for name in ("fig_repeats.png", "fig_prediction.png", "fig_translation.png")]
for figure in figures:
    assert figure.is_file(), figure
    print("figure", figure, figure.stat().st_size)

checks = {
    "M0": (summary["chronological_mae"]["M0"], 0.0583401913),
    "M1": (summary["chronological_mae"]["M1"], 0.0674996752),
    "M2": (summary["chronological_mae"]["M2"], 0.0607088708),
    "M2-M1": (summary["delta_m2_minus_m1"], -0.0067908043),
    "diagnostic M1 alpha 3": (diagnostic["mae_m1_alpha_3"], 0.0594126898),
    "diagnostic M2 alpha 3": (diagnostic["mae_m2_alpha_3"], 0.0607088708),
}
for name, (got, expected) in checks.items():
    print(f"{name}: {got:.10f} (expected {expected:.10f})")
    assert abs(got - expected) < 1e-9, (name, got, expected)
print("primary players", int(ledger["in_primary"].sum()), "of", len(ledger))
print("2025 get-off was not read. This process did not execute on Kaggle.")


In [ ]:
from IPython.display import Image, display
for name in ["fig_repeats.png", "fig_prediction.png", "fig_translation.png"]:
    display(Image(filename=str(WORK_ROOT / "presentation" / name)))
